In [0]:
import os
import sys

# Dynamically construct path to src directory
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
src_path = '/Workspace' + os.path.dirname(os.path.dirname(notebook_path))
sys.path.insert(0, src_path)

import gradio as gr
import pandas as pd

from quattroformaggi.BriefingNoteWriter import brief_writer
from quattroformaggi.QueryInterpreter import interpret_query
from quattroformaggi.query_to_sql import filter_humanitarian_data

# 1. Create the main chat pipeline function
# Gradio requires the function to accept 'message' (current query) and 'history' (chat history)
async def chat_pipeline(message, history):
    try:
        # Step 1: Agent 1 interprets the natural language query
        agent1_result = await interpret_query(message)
        agent_json = agent1_result.model_dump_json()

        # Step 2: Fetch and filter data using Spark
        master_table_path = "unocha.default.master_table"
        
        # We use the global 'spark' object provided by the Databricks environment
        filtered_df = filter_humanitarian_data(agent_json, spark, master_table_path)
        
        # Fallback: Check if the query returned any data
        if filtered_df.count() == 0:
            return "**No Data Found:** Unfortunately, no data matched your criteria. Please try rephrasing your query or broadening the scope."

        # Step 3: Convert the filtered PySpark DataFrame to a Pandas CSV string
        data_as_csv_string = filtered_df.toPandas().to_csv(index=False)

        # Step 4: Agent 2 generates the Briefing Note based on the data
        agent2_result = await brief_writer(data_as_csv_string, agent1_result.interpretation_notes)   
        
        # Return the generated report to the UI
        return agent2_result
        
    except Exception as e:
        # Graceful error handling for the UI
        return f"**Pipeline Error:** An error occurred while processing your request: `{str(e)}`"

# 2. Configure the Gradio Interface
demo = gr.ChatInterface(
    fn=chat_pipeline,
    title="Geo-Insight: GapFinder Assistant",
    description="Ask a natural language question about financial gaps in humanitarian crises. The system will query the underlying data and generate a professional Briefing Note for decision support.",
    examples=[
        "Show underfunded food crises in the Sahel since 2022.",
        "Short term stability forecast for Middle East.",
        "History of funding in Afghanistan."
    ]
)

os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get(scope="datathon-scope", key="ANTHROPIC_API_KEY")

# 3. Launch the app inside Databricks
# inline=True keeps it embedded in the notebook; share=False prevents external public links
demo.launch(share=False, inline=True)